# 04 · Agent Memory
### Practical LLM Engineering — Module 05: LLM Agents

> **Objectives**
> - The four types of agent memory and when to use each
> - Working memory: context window management, trim, and summarise
> - Episodic memory: storing and retrieving past interactions by similarity
> - Semantic memory: embedding-based knowledge base (RAG for agents)
> - Procedural memory: learned action templates from successful episodes
> - Integrated memory-augmented agent with all four systems

---


## 1. Overview

An LLM has no persistent state between API calls. **Memory systems** provide continuity:

```
┌──────────────────────────────────────────────────────────────────┐
│  Working memory   (in-context)                                    │
│  • Current conversation + recent tool results                     │
│  • Hard limit: context window size (~8K–128K tokens)             │
├──────────────────────────────────────────────────────────────────┤
│  Episodic memory  (past events)                                   │
│  • Conversations, task outcomes, errors                           │
│  • Retrieved by semantic similarity to current query             │
├──────────────────────────────────────────────────────────────────┤
│  Semantic memory  (knowledge base)                                │
│  • Facts, documents, domain knowledge                             │
│  • Embedding-based retrieval = RAG for agents                    │
├──────────────────────────────────────────────────────────────────┤
│  Procedural memory  (learned procedures)                          │
│  • Successful action sequences / tool-use patterns                │
│  • Retrieved by task type similarity                              │
└──────────────────────────────────────────────────────────────────┘
```


## 2. Setup

In [ ]:
# !pip install numpy matplotlib scikit-learn -q
import os, re, json, time, random
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Any
from collections import defaultdict
from sklearn.preprocessing import normalize

class MockLLM:
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system, user, max_tokens=512, temperature=0.0):
        time.sleep(0.01)
        text = f"Summary of conversation so far: {user[:80].strip()}... [condensed]"
        return type("R", (), {"text": text, "total_tokens": len((system+user).split())})()
    def __call__(self, s, u, **kw): return self.complete(s, u, **kw)

class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        words = text.lower().split()
        if not words: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(w)] for w in words], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts], dtype=np.float32))

llm      = MockLLM()
embedder = MockEmbedder(dim=64)
print("✅ Setup ready")


## 3. Working Memory

In [ ]:
@dataclass
class Message:
    role:      str
    content:   str
    timestamp: float = field(default_factory=time.time)
    tokens:    int   = 0
    def __post_init__(self):
        if not self.tokens: self.tokens = len(self.content.split())


class WorkingMemory:
    """In-context memory manager with automatic trimming and summarisation."""
    def __init__(self, max_tokens=2000, system_prompt="You are a helpful assistant."):
        self.max_tokens    = max_tokens
        self.system_prompt = system_prompt
        self._messages:  list[Message] = []
        self._summaries: list[str]     = []

    def add(self, role, content):
        self._messages.append(Message(role=role, content=content))
        self._trim()

    def _trim(self):
        while self.token_count > self.max_tokens and len(self._messages) > 2:
            evicted = self._messages.pop(0)
            self._summaries.append(f"[{evicted.role}]: {evicted.content[:80]}...")

    def summarise(self, llm):
        """Compress current messages into a single summary, then flush."""
        ctx  = "\n".join(f"{m.role}: {m.content[:100]}" for m in self._messages)
        resp = llm("Summarise this conversation concisely.",
                    f"Conversation:\n{ctx}\n\nSummary:")
        self._summaries.append(resp.text)
        self._messages.clear()
        return resp.text

    def to_messages(self):
        msgs = []
        if self._summaries:
            msgs.append({"role": "user",      "content": "Prior context: " + " | ".join(self._summaries[-3:])})
            msgs.append({"role": "assistant", "content": "Understood."})
        msgs.extend({"role": m.role, "content": m.content} for m in self._messages)
        return msgs

    @property
    def token_count(self): return sum(m.tokens for m in self._messages)

    def __repr__(self):
        return (f"WorkingMemory(msgs={len(self._messages)}, "
                f"tokens={self.token_count}/{self.max_tokens}, "
                f"summaries={len(self._summaries)})")


# Demo: fill and auto-trim
wm = WorkingMemory(max_tokens=150)
convos = [
    ("user",      "What is RAG and how does it work?"),
    ("assistant", "RAG stands for Retrieval-Augmented Generation. It grounds LLM outputs in retrieved documents to reduce hallucination."),
    ("user",      "What are the main components of a RAG pipeline?"),
    ("assistant", "The main components are: a document store, a retriever, a context assembler, and an LLM generator."),
    ("user",      "How does chunking affect RAG quality?"),
    ("assistant", "Smaller chunks (128-256 tokens) enable precise retrieval but may lose context. Larger chunks preserve narrative but dilute embedding signal."),
]

print("Working memory growth and trimming:\n")
for role, content in convos:
    wm.add(role, content)
    print(f"  After '{content[:38]}...'")
    print(f"    → {wm}")

print(f"\nMessages retained: {len(wm._messages)}")
print(f"Evicted (as summaries): {len(wm._summaries)}")
print("\nFull message list:")
for m in wm._messages:
    print(f"  [{m.role}] {m.content[:60]}...")


## 4. Episodic Memory

In [ ]:
@dataclass
class Episode:
    id:        str
    task:      str
    outcome:   str
    steps:     list
    timestamp: float = field(default_factory=time.time)
    tags:      list  = field(default_factory=list)
    success:   bool  = True
    n_steps:   int   = 0

    def to_text(self):
        return f"Task: {self.task}. Outcome: {self.outcome}. Tags: {', '.join(self.tags)}."


class EpisodicMemory:
    """Stores past agent episodes; retrieves them by semantic similarity."""
    def __init__(self, embedder, max_episodes=500):
        self.embedder     = embedder
        self.max_episodes = max_episodes
        self._episodes:   list[Episode]        = []
        self._vecs:       Optional[np.ndarray] = None

    def store(self, episode):
        self._episodes.append(episode)
        if len(self._episodes) > self.max_episodes: self._episodes.pop(0)
        self._vecs = self.embedder.embed([e.to_text() for e in self._episodes])

    def recall(self, query, k=3, filter_success=None):
        if not self._episodes or self._vecs is None: return []
        q     = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        top_k = np.argsort(-scores)[:k*3]
        results = []
        for i in top_k:
            ep = self._episodes[i]
            if filter_success is not None and ep.success != filter_success: continue
            results.append((ep, float(scores[i])))
            if len(results) >= k: break
        return results

    def success_rate(self):
        if not self._episodes: return 0.0
        return sum(e.success for e in self._episodes) / len(self._episodes)


# Populate with simulated past episodes
mem = EpisodicMemory(embedder)
episodes = [
    Episode("ep001", "Calculate compound interest at 5% for 10 years",
            "Used calculator. Result: £16,288.95", [],
            tags=["math","finance"], success=True, n_steps=2),
    Episode("ep002", "Research transformer attention mechanisms",
            "Retrieved 3 articles. Summarised self-attention and multi-head variants.", [],
            tags=["research","ml"], success=True, n_steps=3),
    Episode("ep003", "Write a Python quicksort implementation",
            "Wrote and tested recursive quicksort.", [],
            tags=["coding","python"], success=True, n_steps=2),
    Episode("ep004", "Find current stock price for AAPL",
            "Failed: no stock price tool available.", [],
            tags=["finance","api"], success=False, n_steps=1),
    Episode("ep005", "Compute the area of a circle with radius 7",
            "calculator: pi * 7^2 = 153.94", [],
            tags=["math","geometry"], success=True, n_steps=1),
]
for ep in episodes: mem.store(ep)

print(f"Episodic memory: {len(mem._episodes)} episodes (success rate: {mem.success_rate():.0%})\n")

queries = ["compute a mathematical formula", "write code in Python", "look up live market data"]
for q in queries:
    recalled = mem.recall(q, k=2, filter_success=True)
    print(f"Query: '{q}'")
    for ep, sc in recalled:
        print(f"  [{ep.id}] sim={sc:.3f}  {ep.task[:55]}")
    print()


## 5. Semantic Memory

In [ ]:
@dataclass
class KnowledgeEntry:
    id:         str
    content:    str
    source:     str
    tags:       list  = field(default_factory=list)
    confidence: float = 1.0


class SemanticMemory:
    """Long-term knowledge base with embedding retrieval — RAG for agents."""
    def __init__(self, embedder):
        self.embedder  = embedder
        self._entries: list[KnowledgeEntry] = []
        self._vecs:    Optional[np.ndarray] = None

    def learn(self, entry):
        self._entries.append(entry)
        self._vecs = self.embedder.embed([e.content for e in self._entries])

    def recall(self, query, k=3, min_confidence=0.0):
        if not self._entries or self._vecs is None: return []
        q      = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        top    = np.argsort(-scores)[:k*2]
        return [(self._entries[i], float(scores[i])) for i in top
                if self._entries[i].confidence >= min_confidence][:k]

    def update_confidence(self, entry_id, delta):
        for e in self._entries:
            if e.id == entry_id:
                e.confidence = max(0.0, min(1.0, e.confidence + delta))

    def forget(self, entry_id):
        self._entries = [e for e in self._entries if e.id != entry_id]
        if self._entries:
            self._vecs = self.embedder.embed([e.content for e in self._entries])


sem = SemanticMemory(embedder)
knowledge = [
    KnowledgeEntry("k001", "Self-attention scales Q·K dot products by sqrt(d_k) before softmax.", "transformers_paper", ["ml","attention"]),
    KnowledgeEntry("k002", "BM25 outperforms TF-IDF for sparse retrieval in most IR benchmarks.", "ir_survey", ["retrieval","bm25"]),
    KnowledgeEntry("k003", "Compound interest: P*(1+r)^n where P=principal, r=rate, n=periods.", "finance_guide", ["math","finance"]),
    KnowledgeEntry("k004", "HNSW builds a hierarchical proximity graph with ef_search controlling recall-speed.", "faiss_docs", ["retrieval","ann"]),
    KnowledgeEntry("k005", "Python list comprehension [f(x) for x in lst] is faster than map().", "python_guide", ["coding","python"]),
]
for k in knowledge: sem.learn(k)

print(f"Semantic memory: {len(sem._entries)} entries\n")
for q in ["how does attention work", "fast nearest neighbour search", "calculate interest"]:
    recalled = sem.recall(q, k=2)
    print(f"Q: '{q}'")
    for entry, sc in recalled:
        print(f"  [{entry.id}] sim={sc:.3f}  {entry.content[:70]}")
    print()


## 6. Procedural Memory

In [ ]:
@dataclass
class ActionTemplate:
    """A learned procedure: when to use it and what steps to take."""
    id:            str
    task_pattern:  str
    steps:         list
    success_count: int   = 0
    fail_count:    int   = 0
    avg_steps:     float = 0.0

    @property
    def reliability(self):
        n = self.success_count + self.fail_count
        return self.success_count / n if n > 0 else 0.5

    def to_text(self):
        return f"Pattern: {self.task_pattern}. Steps: {len(self.steps)}. Reliability: {self.reliability:.0%}."

    def record(self, success, n_steps):
        if success: self.success_count += 1
        else:       self.fail_count    += 1
        n = self.success_count + self.fail_count
        self.avg_steps = (self.avg_steps * (n-1) + n_steps) / n


class ProceduralMemory:
    """Stores learned action templates; retrieves the best match for a new task."""
    def __init__(self, embedder):
        self.embedder    = embedder
        self._templates: list[ActionTemplate]  = []
        self._vecs:      Optional[np.ndarray]  = None

    def store(self, template):
        for i, t in enumerate(self._templates):
            if t.id == template.id:
                self._templates[i] = template; self._rebuild(); return
        self._templates.append(template); self._rebuild()

    def _rebuild(self):
        if self._templates:
            self._vecs = self.embedder.embed([t.to_text() for t in self._templates])

    def retrieve(self, task, k=2, min_reliability=0.4):
        if not self._templates or self._vecs is None: return []
        q      = self.embedder.embed([task])[0]
        scores = self._vecs @ q
        top    = np.argsort(-scores)[:k*3]
        return [(self._templates[i], float(scores[i])) for i in top
                if self._templates[i].reliability >= min_reliability][:k]


proc = ProceduralMemory(embedder)
templates = [
    ActionTemplate("proc_math", "mathematical calculation or formula",
                   [{"tool":"calculator","args":{"expression":"{expr}"}}],
                   success_count=14, fail_count=2, avg_steps=1.5),
    ActionTemplate("proc_research", "research or information lookup",
                   [{"tool":"search","args":{"query":"{topic}"}},
                    {"tool":"search","args":{"query":"{topic} examples"}}],
                   success_count=9, fail_count=3, avg_steps=3.0),
    ActionTemplate("proc_code", "write or execute Python code",
                   [{"tool":"python","args":{"code":"{code}"}}],
                   success_count=17, fail_count=1, avg_steps=2.0),
]
for t in templates: proc.store(t)

print("Procedural memory — template retrieval:\n")
for task in ["compute cube root of 8000", "look up gradient descent algorithms",
              "write a binary search function"]:
    matches = proc.retrieve(task, k=1)
    if matches:
        tmpl, sc = matches[0]
        print(f"  Task: {task[:50]}")
        print(f"  → [{tmpl.id}] reliability={tmpl.reliability:.0%} sim={sc:.3f} steps={[s['tool'] for s in tmpl.steps]}")
    print()


## 7. Integrated Memory Agent

In [ ]:
class MemoryAgent:
    """Agent combining all four memory types."""
    def __init__(self, llm, embedder, working_max_tokens=2000):
        self.llm        = llm
        self.working    = WorkingMemory(working_max_tokens)
        self.episodic   = EpisodicMemory(embedder)
        self.semantic   = SemanticMemory(embedder)
        self.procedural = ProceduralMemory(embedder)
        self.embedder   = embedder

    def _memory_context(self, task):
        parts = []
        # Episodic: past similar successful tasks
        eps = self.episodic.recall(task, k=2, filter_success=True)
        if eps:
            parts.append("Similar past tasks:")
            for ep, _ in eps: parts.append(f"  • {ep.task[:50]} → {ep.outcome[:50]}")
        # Semantic: relevant knowledge
        sems = self.semantic.recall(task, k=2)
        if sems:
            parts.append("Relevant knowledge:")
            for entry, _ in sems: parts.append(f"  • {entry.content[:75]}")
        # Procedural: suggested template
        procs = self.procedural.retrieve(task, k=1)
        if procs:
            tmpl, _ = procs[0]
            parts.append(f"Suggested template: {tmpl.id} ({tmpl.reliability:.0%} reliable) — steps: {[s['tool'] for s in tmpl.steps]}")
        return "\n".join(parts)

    def run(self, task, verbose=True):
        mem_ctx = self._memory_context(task)
        self.working.add("user", task)
        system  = "You are a capable agent with memory. Use past experience to guide your answer."
        user    = f"Memory context:\n{mem_ctx}\n\nTask: {task}" if mem_ctx else f"Task: {task}"
        if verbose:
            print(f"Task: {task}")
            if mem_ctx: print(f"Memory context:\n{mem_ctx}")
        resp   = self.llm(system, user, max_tokens=200)
        answer = resp.text
        self.working.add("assistant", answer)
        self.episodic.store(Episode(
            id=f"ep_{int(time.time()*1000) % 100000}", task=task, outcome=answer[:100],
            steps=[], success=True, n_steps=1,
            tags=re.findall(r"\b\w{5,}\b", task.lower())[:4],
        ))
        if verbose: print(f"Answer: {answer[:120]}...\n")
        return {"task": task, "answer": answer}


# Seed and run
agent = MemoryAgent(llm, embedder)
for ep in episodes[:4]: agent.episodic.store(ep)
for k in knowledge[:3]: agent.semantic.learn(k)
for t in templates[:2]: agent.procedural.store(t)

for task in ["compute factorial of 8", "find information about BM25"]:
    agent.run(task, verbose=True)

print(f"Working memory: {agent.working}")
print(f"Episodic memory: {len(agent.episodic._episodes)} episodes")


## 8. Memory Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Agent Memory System Overview", fontsize=12)

# Working memory token usage
tokens = [m.tokens for m in agent.working._messages]
roles  = [m.role for m in agent.working._messages]
colors = ["#3498db" if r=="user" else "#e67e22" for r in roles]
axes[0].bar(range(len(tokens)), tokens, color=colors)
axes[0].set_title(f"Working Memory\n({agent.working.token_count}/{agent.working.max_tokens} tokens)")
axes[0].set_xlabel("Message index"); axes[0].set_ylabel("Tokens")
# Legend
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="#3498db",label="user"), Patch(color="#e67e22",label="assistant")])
axes[0].grid(axis="y", alpha=0.3)

# Episodic: cumulative success rate
n   = len(agent.episodic._episodes)
csr = [(sum(e.success for e in agent.episodic._episodes[:i+1])/(i+1)) for i in range(n)]
axes[1].plot(range(1,n+1), csr, "o-", color="#2ecc71", lw=2, markersize=6)
axes[1].axhline(0.8, color="gray", linestyle="--", alpha=0.5, label="0.8 target")
axes[1].set_ylim(0,1.05); axes[1].set_title(f"Episodic Memory\n({n} episodes, {agent.episodic.success_rate():.0%} success)")
axes[1].set_xlabel("Episode #"); axes[1].set_ylabel("Cumulative success rate")
axes[1].legend(); axes[1].grid(alpha=0.3)

# Procedural: reliability bars
if agent.procedural._templates:
    names = [t.id for t in agent.procedural._templates]
    rels  = [t.reliability for t in agent.procedural._templates]
    cols  = ["#2ecc71" if r>=0.7 else "#f39c12" for r in rels]
    axes[2].barh(names, rels, color=cols)
    axes[2].axvline(0.7, color="red", linestyle="--", alpha=0.5, label="0.7 threshold")
    axes[2].set_xlim(0, 1.05); axes[2].set_title("Procedural Memory\nTemplate reliability")
    axes[2].set_xlabel("Reliability"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 9. Memory Engineering Considerations

### When to use each memory type

| Memory type | Use when | Avoid when |
|---|---|---|
| Working | All tasks (always present) | Context window is exhausted → summarise |
| Episodic | Agent has many repeated task types | Tasks are always unique |
| Semantic | Domain knowledge is large and stable | Knowledge changes frequently |
| Procedural | Clear task→action patterns exist | Tasks are highly diverse |

### Production engineering

| Challenge | Solution |
|---|---|
| Context overflow | Trim working memory; summarise old turns |
| Memory staleness | Apply confidence decay over time |
| Sensitive data | Never persist PII in episodic/semantic memory |
| Memory poisoning | Validate all stored entries |
| Cold start | Pre-populate semantic memory from documentation |
| Multi-user | Isolate episodic/procedural per user |
